Hypothesis
-----
Does per-neuron gradient variance differ between an overfitting model and a
generalizing (regularized) model? **This notebook replicates the protocol
from `Neuron_Gradient_Variance_Experiments.ipynb` on a real-world,
multi-class, image-like dataset, as part of a multi-dataset validation
sweep for that claim.**

Setup
-----
- Data: sklearn `load_digits` (1797 8x8 grayscale digit images, 64 pixel
  features, 10 classes). Unlike the binary `make_moons` baseline, this tests
  whether the effect generalizes to a multi-class softmax setting with
  `CrossEntropyLoss` instead of binary `BCEWithLogitsLoss`.
- A small, fixed training subset (100 samples, stratified across the 10
  digit classes) is carved out so the wide MLP can genuinely overfit; a
  separate, disjoint, larger validation subset (400 samples) is held out
  from the same pool.
- Pixel features are standardized (`StandardScaler`) using statistics from
  the training subset only, then applied to both splits.
- Model A ("overfit"): no regularization, no dropout, trained blindly for
  all `N_EPOCHS` epochs, sees no validation signal during training.
- Model B ("regularized"): identical architecture, but with dropout + L2
  weight decay, trained for the same number of epochs (not early-stopped)
  so the epoch axes line up for direct comparison.
- Both models use IDENTICAL weight initialization (same seed, same
  architecture) so neuron index N in model A and neuron index N in model B
  start out as literally the same neuron.
- 10% of TRAINING labels are flipped to a random OTHER class (the
  multi-class analogue of the binary label flip in the moons baseline) to
  inject idiosyncratic noise that can only be fit by memorization.
- We log the L2 norm and full gradient vector flowing into each probed
  neuron's incoming weight vector (fc1, 64-dimensional here) at every epoch.

Findings
-----
Across 5 random seeds (42, 7, 13, 99, 2024) the per-neuron gradient analysis
strongly and consistently supports the hypothesis on this multi-class image-like
dataset (10-class `CrossEntropyLoss`, 64 pixel features).

**Dead-neuron threshold was auto-selected per run** (smallest value in
[0.000001, 0.01] giving 5–15 dead neurons post-epoch-200).

| Seed | Dead | Valid | Direction disagree | Magnitude volatile |
|------|------|-------|-------------------|-------------------|
| 42   | 6    | 122   | 87 / 122 (71.3%)  | 110 / 122 (90.2%) |
| 7    | 5    | 123   | 86 / 123 (69.9%)  | 116 / 123 (94.3%) |
| 13   | 5    | 123   | 98 / 123 (79.7%)  | 119 / 123 (96.7%) |
| 99   | 5    | 123   | 97 / 123 (78.9%)  | 101 / 123 (82.1%) |
| 2024 | 5    | 123   | 98 / 123 (79.7%)  | 115 / 123 (93.5%) |
| **Mean ± SD** | | | **93.2 ± 6.1 neurons** | **112.2 ± 6.9 neurons** |

**Summary:** The dead-neuron count was remarkably stable (5–6 across all seeds).
Direction disagreement and magnitude volatility were both elevated in ~70–80%
and ~82–97% of valid neurons respectively, with very low seed-to-seed variance.
This is the cleanest replication of the hypothesis across all datasets in this
sweep, suggesting the gradient-variance effect is particularly robust in multi-
class, image-like settings.

Next Steps
-----
- Compare these stats against the other datasets in this validation sweep
  (`make_moons`, breast cancer, diabetes regression, synthetic
  high-dimensional `make_classification`) to see whether the effect is
  dataset-general or an artifact of the original synthetic/binary setup.


In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import json
import matplotlib.pyplot as plt


In [ ]:
SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

N_EPOCHS = 1000
HIDDEN = 128            # very wide relative to dataset size -> easy to overfit
N_TRAIN = 100           # small training set -> easy to overfit
N_VAL = 400             # disjoint validation pool, drawn from the same data
LABEL_FLIP_FRAC = 0.1   # fraction of TRAIN labels flipped to a random OTHER class
NEURON_LAYER = "fc1"    # which layer we probe (first hidden layer, width=HIDDEN)
N_PROBE_NEURONS = 128
NUM_CLASSES = 10


In [ ]:
# ----------------------------------------------------------------------------
# 1. Data
# ----------------------------------------------------------------------------
# Real-world, image-like data (8x8 handwritten digits): 1797 samples, 64
# pixel features, 10 classes. We carve out a small stratified training
# subset and a disjoint, larger validation subset from the same pool.
X_full, y_full = load_digits(return_X_y=True)

X_train, X_rest, y_train, y_rest = train_test_split(
    X_full, y_full, train_size=N_TRAIN, random_state=SEED, stratify=y_full
)
X_val, _, y_val, _ = train_test_split(
    X_rest, y_rest, train_size=N_VAL, random_state=SEED, stratify=y_rest
)

# Standardize using TRAIN statistics only, then apply to both splits.
scaler = StandardScaler().fit(X_train)
X_train = scaler.transform(X_train)
X_val = scaler.transform(X_val)

# Inject label noise into the TRAINING set only: reassign a fraction of
# training labels to a random OTHER class (multi-class analogue of the
# binary flip in the moons baseline) so the overfit model has concrete
# idiosyncratic points it can only fit by memorizing.
rng_flip = np.random.default_rng(SEED)
n_flip = int(LABEL_FLIP_FRAC * len(y_train))
flip_idx = rng_flip.choice(len(y_train), size=n_flip, replace=False)
y_train = y_train.copy()
for i in flip_idx:
    other_classes = [c for c in range(NUM_CLASSES) if c != y_train[i]]
    y_train[i] = rng_flip.choice(other_classes)
print(f"Flipped {n_flip}/{len(y_train)} training labels to inject memorizable noise.")

X_train_t = torch.tensor(X_train, dtype=torch.float32)
y_train_t = torch.tensor(y_train, dtype=torch.long)
X_val_t = torch.tensor(X_val, dtype=torch.float32)
y_val_t = torch.tensor(y_val, dtype=torch.long)

IN_FEATURES = X_train.shape[1]
print(f"Train size: {len(X_train)}, Val size: {len(X_val)}, in_features: {IN_FEATURES}")


In [ ]:
# ----------------------------------------------------------------------------
# 2. Model definition (identical architecture for both models)
# ----------------------------------------------------------------------------
# Note: fc3 now outputs NUM_CLASSES logits (multi-class) instead of a single
# binary logit -- everything else mirrors the moons-baseline architecture.
class MLP(nn.Module):
    def __init__(self, in_features, hidden=HIDDEN, dropout_p=0.0, out_features=NUM_CLASSES):
        super().__init__()
        self.fc1 = nn.Linear(in_features, hidden)
        self.act1 = nn.ReLU()
        self.drop1 = nn.Dropout(dropout_p) if dropout_p > 0 else nn.Identity()
        self.fc2 = nn.Linear(hidden, hidden)
        self.act2 = nn.ReLU()
        self.drop2 = nn.Dropout(dropout_p) if dropout_p > 0 else nn.Identity()
        self.fc3 = nn.Linear(hidden, out_features)

    def forward(self, x):
        x = self.act1(self.fc1(x))
        x = self.drop1(x)
        x = self.act2(self.fc2(x))
        x = self.drop2(x)
        x = self.fc3(x)
        return x


def make_model(dropout_p):
    torch.manual_seed(SEED)  # identical init for both models
    return MLP(in_features=IN_FEATURES, hidden=HIDDEN, dropout_p=dropout_p)


model_overfit = make_model(dropout_p=0.0)
model_reg = make_model(dropout_p=0.0)

# sanity check: identical initial weights on fc1
assert torch.allclose(model_overfit.fc1.weight, model_reg.fc1.weight)
print("Confirmed: both models start with identical fc1 weights.")

# Pick fixed neuron indices in fc1 (same neurons probed in both models)
rng = np.random.default_rng(SEED)
probe_indices = sorted(rng.choice(HIDDEN, size=N_PROBE_NEURONS, replace=False).tolist())
print(f"Probing neuron indices in {NEURON_LAYER}: {probe_indices}")


In [ ]:
# ----------------------------------------------------------------------------
# 3. Training loop with per-neuron gradient logging
# ----------------------------------------------------------------------------
# Note: CrossEntropyLoss (multi-class) replaces BCEWithLogitsLoss, and
# accuracy is computed via argmax instead of a 0.5 sigmoid threshold --
# everything else mirrors the moons-baseline training loop.
def train_and_log(model, weight_decay, label):
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=1e-3, weight_decay=weight_decay)

    grad_log = {idx: [] for idx in probe_indices}
    grad_vec_log = {idx: [] for idx in probe_indices}
    train_losses, val_losses = [], []
    train_accs, val_accs = [], []

    target_layer_weight = dict(model.named_parameters())[f"{NEURON_LAYER}.weight"]

    for epoch in range(N_EPOCHS):
        model.train()
        optimizer.zero_grad()
        logits = model(X_train_t)
        loss = criterion(logits, y_train_t)
        loss.backward()

        # log gradient norm and full gradient vector for each probed
        # neuron's incoming weight vector
        # target_layer_weight.grad has shape [HIDDEN, IN_FEATURES] for fc1
        grad = target_layer_weight.grad.detach()
        for idx in probe_indices:
            neuron_grad_norm = grad[idx].norm().item()
            grad_log[idx].append(neuron_grad_norm)
            grad_vec_log[idx].append(grad[idx].tolist())

        optimizer.step()

        with torch.no_grad():
            train_pred = logits.argmax(dim=1)
            train_acc = (train_pred == y_train_t).float().mean().item()
            train_losses.append(loss.item())
            train_accs.append(train_acc)

            model.eval()
            val_logits = model(X_val_t)
            val_loss = criterion(val_logits, y_val_t).item()
            val_pred = val_logits.argmax(dim=1)
            val_acc = (val_pred == y_val_t).float().mean().item()
            val_losses.append(val_loss)
            val_accs.append(val_acc)

        if epoch % 100 == 0 or epoch == N_EPOCHS - 1:
            print(
                f"[{label}] epoch {epoch:4d} | train_loss {loss.item():.4f} "
                f"val_loss {val_loss:.4f} | train_acc {train_acc:.3f} val_acc {val_acc:.3f}"
            )

    return {
        "grad_log": grad_log,
        "grad_vec_log": grad_vec_log,
        "train_losses": train_losses,
        "val_losses": val_losses,
        "train_accs": train_accs,
        "val_accs": val_accs,
    }


print("\n--- Training OVERFIT model (no regularization) ---")
results_overfit = train_and_log(model_overfit, weight_decay=0.0, label="overfit")

print("\n--- Training REGULARIZED model (dropout + weight decay) ---")
results_reg = train_and_log(model_reg, weight_decay=5e-2, label="regularized")


In [ ]:
# ----------------------------------------------------------------------------
# 4. Save raw results to disk (so plotting can be re-run without retraining)
# ----------------------------------------------------------------------------
filename = f"results_digits_{SEED}.json"
with open(filename, "w") as f:
    json.dump(
        {
            "probe_indices": probe_indices,
            "overfit": results_overfit,
            "regularized": results_reg,
        },
        f,
    )

print("\nSaved results to ")
print(filename)


In [ ]:
with open(f"results_digits_{SEED}.json", "r") as f:
    data = json.load(f)

probe_indices = data["probe_indices"]
overfit = data["overfit"]
reg = data["regularized"]

N_EPOCHS = len(overfit["train_losses"])
epochs = np.arange(N_EPOCHS)

# ----------------------------------------------------------------------------
# Figure 1: train/val loss curves for both models (context for what follows)
# ----------------------------------------------------------------------------
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

axes[0].plot(epochs, overfit["train_losses"], label="train loss", color="#d62728")
axes[0].plot(epochs, overfit["val_losses"], label="val loss", color="#1f77b4")
axes[0].set_title("Overfit model (no regularization)")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Loss")
axes[0].legend()
axes[0].grid(alpha=0.3)

axes[1].plot(epochs, reg["train_losses"], label="train loss", color="#d62728")
axes[1].plot(epochs, reg["val_losses"], label="val loss", color="#1f77b4")
axes[1].set_title("Regularized model (dropout + L2)")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Loss")
axes[1].legend()
axes[1].grid(alpha=0.3)

fig.suptitle(f"Train vs. Validation Loss: Overfit vs. Regularized Model (Digits (multi-class))", fontsize=13)
fig.tight_layout()
fig.savefig(f"digits_loss_curves_{SEED}.png", dpi=150)
plt.close(fig)

# ----------------------------------------------------------------------------
# Figure 2: superimposed per-neuron gradient norm comparison charts
# ----------------------------------------------------------------------------
fig, axes = plt.subplots(len(probe_indices), 1, figsize=(11, 3.2 * len(probe_indices)), sharex=True)

for ax, idx in zip(axes, probe_indices):
    g_overfit = overfit["grad_log"][str(idx)]
    g_reg = reg["grad_log"][str(idx)]

    ax.plot(epochs, g_overfit, label="Overfit model", color="#d62728", alpha=0.85, linewidth=1.1)
    ax.plot(epochs, g_reg, label="Regularized model", color="#1f77b4", alpha=0.85, linewidth=1.1)
    ax.set_title(f"fc1 neuron #{idx} — incoming weight gradient norm")
    ax.set_ylabel("||grad||_2")
    ax.grid(alpha=0.3)
    ax.legend(loc="upper right", fontsize=8)

axes[-1].set_xlabel("Epoch")
fig.suptitle(f"Per-Neuron Gradient Norm Over Training: Overfit vs. Regularized (Digits (multi-class))\n(same fixed neurons, same initialization, both models)", fontsize=13)
fig.tight_layout(rect=[0, 0, 1, 0.96])
fig.savefig(f"digits_neuron_gradients_comparison_{SEED}.png", dpi=150)
plt.close(fig)

# ----------------------------------------------------------------------------
# Figure 3: windowed variance of those same gradients (the actual hypothesis test)
# ----------------------------------------------------------------------------
WINDOW = 20

def windowed_variance(series, window):
    series = np.array(series)
    out = np.full_like(series, np.nan, dtype=float)
    for i in range(window, len(series)):
        out[i] = series[i - window : i].var()
    return out

fig, axes = plt.subplots(len(probe_indices), 1, figsize=(11, 3.2 * len(probe_indices)), sharex=True)

for ax, idx in zip(axes, probe_indices):
    g_overfit = overfit["grad_log"][str(idx)]
    g_reg = reg["grad_log"][str(idx)]

    wv_overfit = windowed_variance(g_overfit, WINDOW)
    wv_reg = windowed_variance(g_reg, WINDOW)

    ax.plot(epochs, wv_overfit, label="Overfit model", color="#d62728", alpha=0.85, linewidth=1.1)
    ax.plot(epochs, wv_reg, label="Regularized model", color="#1f77b4", alpha=0.85, linewidth=1.1)
    ax.set_title(f"fc1 neuron #{idx} — windowed (w={WINDOW}) gradient variance")
    ax.set_ylabel("Var(grad norm)")
    ax.grid(alpha=0.3)
    ax.legend(loc="upper right", fontsize=8)

axes[-1].set_xlabel("Epoch")
fig.suptitle(f"Windowed Gradient Variance (window={WINDOW} epochs): Overfit vs. Regularized (Digits (multi-class))", fontsize=13)
fig.tight_layout(rect=[0, 0, 1, 0.96])
fig.savefig(f"digits_neuron_gradient_variance_comparison_{SEED}.png", dpi=150)
plt.close(fig)

print("Saved: digits_loss_curves.png, digits_neuron_gradients_comparison.png, digits_neuron_gradient_variance_comparison.png")


In [ ]:
# ----------------------------------------------------------------------------
# Auto-select DEAD_THRESHOLD: scan candidate thresholds between 0.0001 and
# 0.01 and pick the SMALLEST one that classifies between 5 and 15 neurons as
# dead (post-epoch-200 mean grad norm below threshold, in EITHER model --
# same rule used everywhere below). A fixed magic-number threshold doesn't
# transfer across datasets/runs whose gradients converge to different
# absolute scales, so we calibrate it fresh against this run's own data.
# ----------------------------------------------------------------------------
def count_dead(threshold):
    count = 0
    for idx in probe_indices:
        mean_overfit = np.array(overfit["grad_log"][str(idx)])[200:].mean()
        mean_reg = np.array(reg["grad_log"][str(idx)])[200:].mean()
        if mean_overfit < threshold or mean_reg < threshold:
            count += 1
    return count

candidate_thresholds = np.logspace(np.log10(0.0001), np.log10(0.01), 500)

DEAD_THRESHOLD = None
for t in candidate_thresholds:
    n_dead = count_dead(t)
    if 5 <= n_dead <= 15:
        DEAD_THRESHOLD = t
        break

if DEAD_THRESHOLD is None:
    DEAD_THRESHOLD = 0.0001
    print(
        f"WARNING: no threshold in [0.0001, 0.01] gave 5-15 dead neurons "
        f"({count_dead(0.0001)} dead at 0.0001, {count_dead(0.01)} dead at 0.01). "
        f"Falling back to DEAD_THRESHOLD=0.0001 -- inspect the dead-neuron "
        f"count curve manually."
    )
else:
    print(f"Auto-selected DEAD_THRESHOLD = {DEAD_THRESHOLD:.6f} "
          f"({count_dead(DEAD_THRESHOLD)} dead neurons)")

dead_neurons = []
n_valid = 0

for idx in probe_indices:
    g_overfit = np.array(overfit["grad_log"][str(idx)])
    g_reg = np.array(reg["grad_log"][str(idx)])

    post_200_overfit = g_overfit[200:]
    post_200_reg = g_reg[200:]

    mean_overfit = post_200_overfit.mean()
    mean_reg = post_200_reg.mean()

    # exclude dead neurons: either model's mean grad below threshold
    if mean_overfit < DEAD_THRESHOLD or mean_reg < DEAD_THRESHOLD:
        dead_neurons.append(idx)
        continue

    n_valid += 1

print(f"Random seed used: {SEED}")
print(f"Dead neurons excluded: {len(dead_neurons)}/{len(probe_indices)} -> {dead_neurons}")
print(f"Valid neurons compared: {n_valid}/{len(probe_indices)}")


In [ ]:
with open(f"results_digits_{SEED}.json", "r") as f:
    data = json.load(f)

probe_indices = data["probe_indices"]
overfit = data["overfit"]
reg = data["regularized"]

num_epochs_logged = len(overfit["train_losses"])
epochs = np.arange(num_epochs_logged)

def cosine_similarity_series(grad_vecs):
    """grad_vecs: list of per-epoch gradient vectors (length = input dim,
    i.e. 2 for the moons baseline, but arbitrary here). Returns cosine
    similarity between each epoch's gradient vector and the PREVIOUS
    epoch's, same length as input (first epoch has no predecessor)."""
    vecs = np.array(grad_vecs)
    sims = np.full(len(vecs), np.nan)
    for t in range(1, len(vecs)):
        v_prev, v_curr = vecs[t - 1], vecs[t]
        norm_prev, norm_curr = np.linalg.norm(v_prev), np.linalg.norm(v_curr)
        if norm_prev > 1e-12 and norm_curr > 1e-12:
            sims[t] = np.dot(v_prev, v_curr) / (norm_prev * norm_curr)
    return sims

# ----------------------------------------------------------------------------
# Filter out dead neurons (same rule as before: either model's mean grad
# norm in the post-200 window below threshold)
# ----------------------------------------------------------------------------
valid_indices = []
for idx in probe_indices:
    mean_overfit = np.array(overfit["grad_log"][str(idx)])[200:].mean()
    mean_reg = np.array(reg["grad_log"][str(idx)])[200:].mean()
    if mean_overfit >= DEAD_THRESHOLD and mean_reg >= DEAD_THRESHOLD:
        valid_indices.append(idx)

print(f"Valid (non-dead) neurons for direction analysis: {len(valid_indices)}/{len(probe_indices)}")

# ----------------------------------------------------------------------------
# Plot 1: cosine similarity to previous epoch, for all valid neurons
# ----------------------------------------------------------------------------
plot_indices = valid_indices

fig, axes = plt.subplots(len(plot_indices), 1, figsize=(11, 3.2 * len(plot_indices)), sharex=True)
for ax, idx in zip(axes, plot_indices):
    sim_overfit = cosine_similarity_series(overfit["grad_vec_log"][str(idx)])
    sim_reg = cosine_similarity_series(reg["grad_vec_log"][str(idx)])

    ax.plot(epochs, sim_overfit, label="Overfit model", color="#d62728", alpha=0.8, linewidth=0.9)
    ax.plot(epochs, sim_reg, label="Generalizing model", color="#1f77b4", alpha=0.8, linewidth=0.9)
    ax.axhline(0, color="gray", linewidth=0.5, linestyle="--")
    ax.set_title(f"fc1 neuron #{idx} — gradient direction consistency (cos sim, epoch t vs t-1)")
    ax.set_ylabel("cosine similarity")
    ax.set_ylim(-1.1, 1.1)
    ax.grid(alpha=0.3)
    ax.legend(loc="lower right", fontsize=8)

axes[-1].set_xlabel("Epoch")
fig.suptitle(
    "Gradient Direction Consistency: does the gradient keep pointing the same way?\n"
    "(1 = same direction as last epoch, 0 = unrelated, -1 = flipped)",
    fontsize=12,
)
fig.tight_layout(rect=[0, 0, 1, 0.95])
fig.savefig(f"digits_direction_consistency_{SEED}.png", dpi=150)
plt.close(fig)

# ----------------------------------------------------------------------------
# Summary stat: average cosine similarity post-epoch-200, across ALL valid
# neurons, for both models -- the direct test of "overfit model disagrees
# with itself over time more than the generalizing model does"
# ----------------------------------------------------------------------------
overfit_lower_consistency = 0
print("Mean Overfit Similarity | Mean Regular Similarity\n")
for idx in valid_indices:
    sim_overfit = cosine_similarity_series(overfit["grad_vec_log"][str(idx)])[200:]
    sim_reg = cosine_similarity_series(reg["grad_vec_log"][str(idx)])[200:]
    mean_sim_overfit = np.nanmean(sim_overfit)
    mean_sim_reg = np.nanmean(sim_reg)
    if mean_sim_overfit < mean_sim_reg:
        overfit_lower_consistency += 1
print(f"\nFor seed = {SEED}")
print(f"Overfit model shows LOWER direction consistency (more disagreement) "
      f"in {overfit_lower_consistency}/{len(valid_indices)} neurons post-epoch-200")

print("Saved: digits_direction_consistency.png")


In [ ]:
def log_ratio_series(grad_norm_series):
    """grad_norm_series: list of per-epoch L2 gradient norms (from grad_log).
    Returns log(g_t / g_t-1) for each epoch t, length matches input (first
    entry is NaN, no predecessor). Scale-free and symmetric -- not
    contaminated by (a) the two models sitting at different absolute
    gradient scales, or (b) a smooth multiplicative trend (e.g. exponential
    decay) within the analysis window. A constant log-ratio = a perfectly
    smooth trend; a swinging log-ratio = genuine local volatility."""
    g = np.array(grad_norm_series)
    log_ratios = np.full(len(g), np.nan)
    valid = g > 0  # guard against log(0) for fully-dead epochs
    log_g = np.full(len(g), np.nan)
    log_g[valid] = np.log(g[valid])
    log_ratios[1:] = log_g[1:] - log_g[:-1]
    return log_ratios


fig, axes = plt.subplots(len(plot_indices), 1, figsize=(11, 3.2 * len(plot_indices)), sharex=True)
for ax, idx in zip(axes, plot_indices):
    lr_overfit = log_ratio_series(overfit["grad_log"][str(idx)])
    lr_reg = log_ratio_series(reg["grad_log"][str(idx)])

    ax.plot(epochs, lr_overfit, label="Overfit model", color="#d62728", alpha=0.8, linewidth=0.9)
    ax.plot(epochs, lr_reg, label="Generalizing model", color="#1f77b4", alpha=0.8, linewidth=0.9)
    ax.axhline(0, color="gray", linewidth=0.5, linestyle="--")
    ax.set_title(f"fc1 neuron #{idx} — magnitude volatility (log-ratio, epoch t vs t-1)")
    ax.set_ylabel("log(g_t / g_t-1)")
    ax.grid(alpha=0.3)
    ax.legend(loc="upper right", fontsize=8)

axes[-1].set_xlabel("Epoch")
fig.suptitle(
    "Gradient Magnitude Volatility: does the gradient's SIZE jump around erratically?\n"
    "(0 = no change, scale-free via log -- a smooth trend collapses to ~constant, not high variance)",
    fontsize=12,
)
fig.tight_layout(rect=[0, 0, 1, 0.95])
fig.savefig(f"digits_magnitude_volatility_{SEED}.png", dpi=150)
plt.close(fig)
print("Saved: digits_magnitude_volatility.png")


In [ ]:
overfit_more_volatile = 0
print("Magnitude volatility (std of log-ratios), Overfit vs Regularized\n")
for idx in valid_indices:
    lr_overfit = log_ratio_series(overfit["grad_log"][str(idx)])[200:]
    lr_reg = log_ratio_series(reg["grad_log"][str(idx)])[200:]
    vol_overfit = np.nanstd(lr_overfit)
    vol_reg = np.nanstd(lr_reg)
    if vol_overfit > vol_reg:
        overfit_more_volatile += 1

print(f"\nFor seed = {SEED}")
print(f"Overfit model shows HIGHER magnitude volatility (more local jumpiness) "
      f"in {overfit_more_volatile}/{len(valid_indices)} neurons post-epoch-200")
